# 01 — Raw Data Audit

Run this notebook **before** `dbt build` to verify the raw CSV files are present and well-formed.

Checks: file presence, row counts, column types, missing values, date ranges, churn label distribution, leakage risk.

Inputs: `data/raw/*.csv` (download via `scripts/download_data.sh`).

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

RAW = Path('../data/raw')
EXPECTED = ['members_v3.csv', 'transactions_v2.csv', 'user_logs_v2.csv', 'train_v2.csv']

con = duckdb.connect(database=':memory:')
con.execute("PRAGMA enable_progress_bar")

## File presence and size

In [ ]:
def file_size_mb(p: Path) -> float:
    return p.stat().st_size / (1024 ** 2) if p.exists() else 0.0

rows = []
for fname in EXPECTED:
    p = RAW / fname
    rows.append({
        'file': fname,
        'exists': p.exists(),
        'size_mb': round(file_size_mb(p), 1),
    })
files_df = pd.DataFrame(rows)
files_df

In [ ]:
missing = [r['file'] for _, r in files_df.iterrows() if not r['exists']]
if missing:
    raise FileNotFoundError(f'Missing raw files: {missing}. Run scripts/download_data.sh first.')
print('All required v2 files present.')

## Row counts

In [ ]:
row_counts = {}
for fname in EXPECTED:
    n = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{RAW / fname}', header=true)").fetchone()[0]
    row_counts[fname] = n
    print(f'{fname:30s} {n:>12,}')

## Column types and sample

In [ ]:
for fname in EXPECTED:
    print(f'\n=== {fname} ===')
    schema = con.execute(
        f"SELECT column_name, column_type FROM (DESCRIBE SELECT * FROM read_csv_auto('{RAW / fname}', header=true) LIMIT 0)"
    ).df()
    print(schema.to_string(index=False))

## Missing values per column (members_v3)

In [ ]:
members = con.execute(f"SELECT * FROM read_csv_auto('{RAW / 'members_v3.csv'}', header=true)").df()
print('Shape:', members.shape)
members.isna().sum().sort_values(ascending=False)

In [ ]:
print('bd (age) distribution — note many erroneous values:')
members['bd'].describe()

## Date ranges (transactions and user_logs)

In [ ]:
txn_dates = con.execute(f"""
    SELECT 
        MIN(strptime(CAST(transaction_date AS VARCHAR), '%Y%m%d')::DATE) AS min_txn_date,
        MAX(strptime(CAST(transaction_date AS VARCHAR), '%Y%m%d')::DATE) AS max_txn_date,
        MIN(strptime(CAST(membership_expire_date AS VARCHAR), '%Y%m%d')::DATE) AS min_expire,
        MAX(strptime(CAST(membership_expire_date AS VARCHAR), '%Y%m%d')::DATE) AS max_expire
    FROM read_csv_auto('{RAW / 'transactions_v2.csv'}', header=true)
""").df()
txn_dates

In [ ]:
log_dates = con.execute(f"""
    SELECT
        MIN(strptime(CAST(date AS VARCHAR), '%Y%m%d')::DATE) AS min_listen_date,
        MAX(strptime(CAST(date AS VARCHAR), '%Y%m%d')::DATE) AS max_listen_date
    FROM read_csv_auto('{RAW / 'user_logs_v2.csv'}', header=true)
""").df()
log_dates

## Churn label distribution

In [ ]:
labels = con.execute(f"SELECT * FROM read_csv_auto('{RAW / 'train_v2.csv'}', header=true)").df()
print('Total labelled users:', len(labels))
label_dist = labels['is_churn'].value_counts(normalize=True).rename({0: 'retained', 1: 'churned'})
label_dist

## Leakage check

We define a feature observation cutoff at `2017-03-01`. Any user_log date or transaction date **after** this cutoff cannot be used as a predictive feature without leakage.

These rows are not removed here — they are filtered out inside the dbt models (`int_user_*` views use `WHERE listen_date <= var('engagement_window_end')` and `WHERE transaction_date <= var('observation_cutoff')`). This check confirms how much data is post-cutoff so we know the impact.

In [ ]:
post_cutoff_logs = con.execute(f"""
    SELECT COUNT(*) AS n
    FROM read_csv_auto('{RAW / 'user_logs_v2.csv'}', header=true)
    WHERE strptime(CAST(date AS VARCHAR), '%Y%m%d')::DATE >= DATE '2017-03-01'
""").fetchone()[0]
post_cutoff_txn = con.execute(f"""
    SELECT COUNT(*) AS n
    FROM read_csv_auto('{RAW / 'transactions_v2.csv'}', header=true)
    WHERE strptime(CAST(transaction_date AS VARCHAR), '%Y%m%d')::DATE > DATE '2017-03-01'
""").fetchone()[0]
print(f'Post-cutoff logs: {post_cutoff_logs:,}')
print(f'Post-cutoff transactions: {post_cutoff_txn:,}')

## Conclusions

If all checks pass:
- All four files present and within expected size ranges.
- Row counts match documented expectations (within ±10%).
- `is_churn` distribution is roughly 90/10 retained/churned.
- Date ranges align with WSDM Cup 2018 documentation.
- Post-cutoff data is correctly filtered by dbt models.

Next: `make dbt-build`.